In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

import os
from pathlib import Path

TASK_NAME = "cogflex_suite_flexible"
FACULTY_ID = "executive_functions/cognitive_flexibility"
TURN_HEADER_PREFIX = "CogFlex suite task. Episode "
EXPECTED_PUBLIC_EPISODE_COUNT = 120
DATASET_ROOT_ENV_VAR = "COGFLEX_DATASET_ROOT"
PRIVATE_DATASET_ROOT_ENV_VAR = "COGFLEX_PRIVATE_DATASET_ROOT"
PRIVATE_ANSWER_KEY_PATH_ENV_VAR = "COGFLEX_PRIVATE_ANSWER_KEY_PATH"
EVAL_SPLIT = os.environ.get("COGFLEX_EVAL_SPLIT", "public").strip().lower()
DEFAULT_DATASET_ROOT = Path("/kaggle/input/datasets/raptorengineer/cogflex-suite-runtime")
DEFAULT_PRIVATE_DATASET_ROOT = Path("/kaggle/input/datasets/raptorengineer/cogflex-suite-runtime-private")
DATASET_ROOT = Path(os.environ.get(DATASET_ROOT_ENV_VAR, str(DEFAULT_DATASET_ROOT)))
PRIVATE_DATASET_ROOT = Path(os.environ.get(PRIVATE_DATASET_ROOT_ENV_VAR, str(DEFAULT_PRIVATE_DATASET_ROOT)))
ROWS_PATH = (DATASET_ROOT / "public_leaderboard_rows.json") if EVAL_SPLIT == "public" else (PRIVATE_DATASET_ROOT / "private_leaderboard_rows.json")
private_answer_key_raw = os.environ.get(PRIVATE_ANSWER_KEY_PATH_ENV_VAR, "").strip()
PRIVATE_ANSWER_KEY_PATH = Path(private_answer_key_raw) if private_answer_key_raw else None


In [ ]:
from dataclasses import is_dataclass

class KaggleExecutionError(RuntimeError):
    pass


In [ ]:
import json

def _normalize_response_spec(spec: object) -> dict[str, object]:
    if not isinstance(spec, dict):
        raise ValueError("response_spec must be a JSON object")
    if spec.get("format") != "ordered_labels":
        raise ValueError("unsupported response format")
    probe_count = spec.get("probe_count")
    label_vocab = spec.get("label_vocab")
    if not isinstance(probe_count, int) or probe_count <= 0:
        raise ValueError("response_spec.probe_count must be positive")
    if not isinstance(label_vocab, list) or len(label_vocab) < 2:
        raise ValueError("response_spec.label_vocab must contain at least two labels")
    normalized = [str(label).strip() for label in label_vocab]
    if len(set(normalized)) != len(normalized):
        raise ValueError("response_spec.label_vocab must be unique")
    return {"format": "ordered_labels", "probe_count": probe_count, "label_vocab": normalized}

def _extract_sequence_response(response: object, probe_count: int) -> tuple[str, ...] | None:
    if isinstance(response, str):
        chunks = [part.strip() for part in response.replace("\n", ",").split(",") if part.strip()]
        return tuple(chunks) if chunks else None
    if isinstance(response, (list, tuple)):
        return tuple(str(item).strip() for item in response)
    if isinstance(response, dict):
        values = []
        for index in range(1, probe_count + 1):
            key = f"probe_{index}"
            if key not in response:
                return None
            values.append(str(response[key]).strip())
        return tuple(values)
    if is_dataclass(response):
        values = []
        for index in range(1, probe_count + 1):
            key = f"probe_{index}"
            if not hasattr(response, key):
                return None
            values.append(str(getattr(response, key)).strip())
        return tuple(values)
    if hasattr(response, "__dict__"):
        return _extract_sequence_response(vars(response), probe_count)
    return None

def normalize_ordered_labels(response: object, response_spec: object) -> tuple[str, ...] | None:
    normalized_spec = _normalize_response_spec(response_spec)
    values = _extract_sequence_response(response, normalized_spec["probe_count"])
    if values is None or len(values) != normalized_spec["probe_count"]:
        return None
    if any(value not in normalized_spec["label_vocab"] for value in values):
        raise ValueError("response contains a label outside label_vocab")
    return values


In [ ]:
def score_episode(predictions: tuple[str, ...] | None, targets: tuple[str, ...]) -> dict:
    if predictions is None:
        predictions = tuple("" for _ in targets)
        numerator = 0
    else:
        numerator = sum(pred == target for pred, target in zip(predictions, targets))
    return {
        "numerator": numerator,
        "denominator": len(targets),
        "predictions": list(predictions),
    }

@kbench.task(store_task=False)
def run_flexible_task(
    llm: object,
    turns: list[str],
    response_spec: dict[str, object],
    final_probe_targets: tuple[str, ...],
) -> dict:
    if len(turns) < 2:
        raise KaggleExecutionError("expected at least two turns")
    try:
        for turn in turns[:-1]:
            llm.prompt(turn)
        response = llm.prompt(turns[-1])
    except Exception as exc:
        raise KaggleExecutionError("llm.prompt failed") from exc
    try:
        normalized = normalize_ordered_labels(response, response_spec)
    except ValueError as exc:
        raise KaggleExecutionError(f"invalid flexible response: {exc}") from exc
    if normalized is None:
        raise KaggleExecutionError(f"unscoreable response of type {type(response).__name__}")
    return score_episode(normalized, final_probe_targets)


In [ ]:
import json
import re
from collections import Counter

LINE_RE = re.compile(r"^(?P<index>\d+)\.\s+(?P<body>.+?)\s+->\s+(?P<label>[a-z0-9_:-]+|\?)$")
POINT_RE = re.compile(r"^r1=(?P<r1>[+-]\d+),\s*r2=(?P<r2>[+-]\d+)$")

def load_selected_rows() -> list[dict[str, object]]:
    return _load_rows(ROWS_PATH)

def attach_selected_scoring(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    if EVAL_SPLIT == "public":
        return rows
    return _attach_private_scoring(rows)

def _parse_case_line(line: str) -> dict[str, object] | None:
    match = LINE_RE.match(line.strip())
    if match is None:
        return None
    payload: dict[str, object] = {"index": int(match.group("index")), "label": match.group("label")}
    point_found = False
    for chunk in (part.strip() for part in match.group("body").split("|")):
        point_match = POINT_RE.match(chunk)
        if point_match is not None:
            payload["r1"] = int(point_match.group("r1"))
            payload["r2"] = int(point_match.group("r2"))
            point_found = True
            continue
        key, value = chunk.split("=", 1)
        payload[key.strip()] = value.strip()
    if not point_found:
        raise ValueError(f"missing coordinates in line: {line!r}")
    return payload

def _parse_turn_items(turn: str, kind: str) -> list[dict[str, object]]:
    items = []
    for line in turn.splitlines():
        parsed = _parse_case_line(line)
        if parsed is None:
            continue
        if kind == "evidence" and parsed["label"] == "?":
            continue
        if kind == "decision" and parsed["label"] != "?":
            continue
        items.append(parsed)
    return items

def _response_instruction(spec: dict[str, object]) -> str:
    vocab = ", ".join(str(label) for label in spec["label_vocab"])
    return f"Return exactly {spec['probe_count']} outputs in order, one per probe. Use only labels from: {vocab}."

def _validate_turn(turn: str, episode_id: str, turn_index: int, total_turns: int, spec: dict[str, object], response_spec: dict[str, object]) -> None:
    if not turn.startswith(f"{TURN_HEADER_PREFIX}{episode_id}. Turn {turn_index} of {total_turns}."):
        raise ValueError(f"episode {episode_id}: malformed header for turn {turn_index}")
    items = _parse_turn_items(turn, str(spec["kind"]))
    if len(items) != int(spec["item_count"]):
        raise ValueError(f"episode {episode_id}: turn {turn_index} count does not match turn_specs")
    if spec["kind"] == "decision":
        if int(spec["item_count"]) != int(response_spec["probe_count"]):
            raise ValueError(f"episode {episode_id}: decision turn does not match probe_count")
        if _response_instruction(response_spec) not in turn:
            raise ValueError(f"episode {episode_id}: decision turn must include the response instruction")
    else:
        for item in items:
            if str(item["label"]) not in response_spec["label_vocab"]:
                raise ValueError(f"episode {episode_id}: evidence turn label is outside label_vocab")

def _normalize_final_probe_targets(values: object, response_spec: dict[str, object]) -> tuple[str, ...]:
    normalized = normalize_ordered_labels(values, response_spec)
    if normalized is None:
        raise ValueError("final_probe_targets must contain exactly probe_count valid labels")
    return normalized

def _validate_row(row: dict[str, object]) -> None:
    episode_id = str(row["episode_id"])
    analysis = row["analysis"]
    inference = row["inference"]
    response_spec = _normalize_response_spec(inference["response_spec"])
    if analysis["faculty_id"] != FACULTY_ID:
        raise ValueError(f"episode {episode_id}: unsupported faculty_id")
    if analysis["difficulty_bin"] not in {"hard", "medium"}:
        raise ValueError(f"episode {episode_id}: unsupported difficulty_bin")
    turns = inference["turns"]
    specs = inference["turn_specs"]
    if not isinstance(turns, list) or not isinstance(specs, list) or len(turns) != len(specs) or len(turns) < 2:
        raise ValueError(f"episode {episode_id}: invalid flexible turn layout")
    if [spec["kind"] for spec in specs].count("decision") != 1 or specs[-1]["kind"] != "decision":
        raise ValueError(f"episode {episode_id}: expected a final decision turn")
    for turn_index, (turn, spec) in enumerate(zip(turns, specs), start=1):
        _validate_turn(turn, episode_id, turn_index, len(turns), spec, response_spec)

def _validate_batch(rows: list[dict[str, object]]) -> None:
    if not rows:
        raise RuntimeError("benchmark dataset must contain at least one row")
    if EVAL_SPLIT == "public" and len(rows) != EXPECTED_PUBLIC_EPISODE_COUNT:
        raise RuntimeError(f"expected {EXPECTED_PUBLIC_EPISODE_COUNT} public rows, found {len(rows)}")
    for row in rows:
        _validate_row(row)

def _load_rows(path: Path) -> list[dict[str, object]]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, list):
        raise RuntimeError(f"expected a JSON list at {path}")
    _validate_batch(payload)
    return payload

def _load_private_answer_key(path: Path) -> dict[str, object]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError("private answer key must be a JSON object")
    if payload.get("split") != "private":
        raise RuntimeError("private answer key must declare split='private'")
    return payload

def _attach_private_scoring(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    if PRIVATE_ANSWER_KEY_PATH is None:
        raise RuntimeError("Private split requires an external answer key")
    answer_key = _load_private_answer_key(PRIVATE_ANSWER_KEY_PATH)
    episodes = {str(item["episode_id"]): item for item in answer_key["episodes"]}
    attached = []
    for row in rows:
        episode = episodes.get(str(row["episode_id"]))
        if episode is None:
            raise RuntimeError(f"missing answer key episode {row['episode_id']}")
        if episode.get("inference") != row["inference"]:
            raise RuntimeError(f"answer key inference mismatch for episode {row['episode_id']}")
        response_spec = _normalize_response_spec(row["inference"]["response_spec"])
        targets = _normalize_final_probe_targets(episode.get("final_probe_targets"), response_spec)
        attached.append({
            "episode_id": row["episode_id"],
            "analysis": dict(row["analysis"]),
            "inference": dict(row["inference"]),
            "scoring": {"final_probe_targets": list(targets)},
        })
    return attached

leaderboard_rows = load_selected_rows()
scored_rows = attach_selected_scoring(leaderboard_rows)
df = pd.DataFrame([
    {
        "turns": row["inference"]["turns"],
        "response_spec": row["inference"]["response_spec"],
        "final_probe_targets": row["scoring"]["final_probe_targets"],
    }
    for row in scored_rows
]) if pd is not None else []


In [ ]:
import json

def summarize_suite_benchmark(runs, rows: list[dict[str, object]]) -> dict[str, object]:
    if not runs:
        raise RuntimeError("evaluation produced no successful runs")
    results_df = runs.as_dataframe().reset_index(drop=True)
    if len(results_df) != len(rows):
        raise RuntimeError(f"evaluation completed {len(results_df)} of {len(rows)} benchmark episodes")
    overall_numerator = 0
    overall_denominator = 0
    per_task: dict[str, dict[str, int]] = {}
    per_structure: dict[str, dict[str, int]] = {}
    per_difficulty: dict[str, dict[str, int]] = {}
    for row, result in zip(rows, results_df.result):
        suite_task_id = row["analysis"]["suite_task_id"]
        structure_family_id = row["analysis"]["structure_family_id"]
        difficulty_bin = row["analysis"]["difficulty_bin"]
        numerator = int(result["numerator"])
        denominator = int(result["denominator"])
        overall_numerator += numerator
        overall_denominator += denominator
        per_task.setdefault(suite_task_id, {"numerator": 0, "denominator": 0})
        per_structure.setdefault(structure_family_id, {"numerator": 0, "denominator": 0})
        per_difficulty.setdefault(difficulty_bin, {"numerator": 0, "denominator": 0})
        per_task[suite_task_id]["numerator"] += numerator
        per_task[suite_task_id]["denominator"] += denominator
        per_structure[structure_family_id]["numerator"] += numerator
        per_structure[structure_family_id]["denominator"] += denominator
        per_difficulty[difficulty_bin]["numerator"] += numerator
        per_difficulty[difficulty_bin]["denominator"] += denominator
    per_task_accuracy = {key: value["numerator"] / value["denominator"] for key, value in sorted(per_task.items())}
    macro_accuracy = sum(per_task_accuracy.values()) / len(per_task_accuracy)
    return {
        "score": macro_accuracy,
        "macro_accuracy": macro_accuracy,
        "micro_accuracy": overall_numerator / overall_denominator,
        "numerator": overall_numerator,
        "denominator": overall_denominator,
        "episodes": len(rows),
        "per_task_accuracy": per_task_accuracy,
        "structure_family_accuracy": {key: value["numerator"] / value["denominator"] for key, value in sorted(per_structure.items())},
        "difficulty_bin_accuracy": {key: value["numerator"] / value["denominator"] for key, value in sorted(per_difficulty.items())},
    }

@kbench.task(name="cogflex_suite_flexible", description="Cognitive flexibility benchmark within executive functions with variable turn structures and label vocabularies.")
def cogflex_suite_flexible(llm) -> float:
    runs = run_flexible_task.evaluate(llm=[llm], evaluation_data=df)
    summary = summarize_suite_benchmark(runs, scored_rows)
    print(json.dumps(summary, indent=2))
    return summary["score"]


In [ ]:
%choose cogflex_suite_flexible
